# 4. SCIKIT-LEARN

## 4.1 Machine Learning Workflow

We have learned NumPy, Pandas and Matplotlib before. These libraries tell us how to deal with data and visualize them. Scikit-learn tells us how to let models learn rules from existing data, and try to predict on unseen data.

### 4.1.1 Standard Machine Learning Workflow

A standard machine learning workflow is consisted of the following parts:

$$
Dataset(x,y) \rightarrow Trainset/Testset \rightarrow Evaluation(Model\ Performance)
$$

The most significant difference here is:

- Train set is used to learn parameters
- Test set is used to evaluate the capability of generalization of the model.


### 4.1.2 Dataset - Feature & Target

First, we use the classic Iris dataset. It has 150 examples, 4 features and 3 classes.


In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()

X = iris.data
y = iris.target

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target names:", iris.target_names)

in which:

- X is a two-dimentional array, $X.shape = (n_{samples}, \ n_{features})$.
- y is a one-dimentional array, because each sample has only one Target Label.


### 4.1.3 Train Test Split

Scikit-learn offers train test split method to better evaluate the generalization ability of the model:


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,  # split 30% of the samples into the Testset;
    random_state=42,  # set fixed random seed, so the split can be re-realized;
    shuffle=True,  # Set as default, to random the order of samples;
    stratify=y,  # Remain the class proportions of Trainset and Testset.
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

However, always notice that for time-ordered data like stocks, we often have to set `shuffle` as `False`.


### 4.1.4 Core of Scikit-Learn: Estimator API

The Machine Learning Models in scikit-learn are often called Estimators. The port is unified:


In [ ]:
# model = SomeEstimator(...)
# model.fit(X_train, y_train)
# y_pred = model.predict(X_test)

in which:

- `.fit()` asks models to learn from Training Data;
- `.predict()` uses testsets to predict answers;
- `.score()` uses the default scoring method of the Estimator to evaluate the ability.

(Notice that the meaning of `score()` relies on the model used. For example, Classifier uses Accuracy, while Regressor uses $R^2$)


Let's first take KNN (K-Nearest Neighbors) as an example:


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=3)  # n_neighbors is a Hyperparameter.
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy = accuracy_score(
    y_test, y_pred
)  # we should know that accuracy may not be the only evaluation method.

print("Accuracy:", accuracy)

### 4.1.5 Feature Scaling & Data Leakage

We have learned Standardization before. In sklearn, this can be done with StandardScaler:


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

in which:

- `fit()`: learn Preprocessing Parameters (Mean, Standard-Deviation, etc.) from data;
- `transform()`: use Parameters already learned to transform the data;
- `fit_transform()`: combine the two process upon, fit first and then transform.

=> Notice that Testset only used `transform()`, or the data from testset will be used in the fitting process, which causes data leakage.

One recommended method is to use `Pipeline`:


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=3),
)

model.fit(X_train, y_train)

print("Test Accuracy:", model.score(X_test, y_test))

## 4.2 Validation Sets, Cross Validation & Evaluation Metrics

### 4.2.1 Three Different Sets

If we directly compare the results with different Hyperparameters, the Testset will participate in the model selection process, which might cause the test accuracy being too optimistic. Thus, we often need an independent dataset to compare models. This dataset is called Validation Sets.

Thus, we can say:

- Training Set: learn model parameters, used for `fit()`;
- Validation Set: model selection and Hyperparameter Tuning, used for model comparation;
- Test Set: final generalization evaluation, used for final performance output.

Also be careful about:

- Model Parameters: parameters learned from training data;
- Hyperparameters: parameters set before training (e.g. `n_neighbors` from KNN).


### 4.2.2 Creating a Validation Set

We can simply use the `train_test_split()` twice to establish the three sets:


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()

X = iris.data
y = iris.target

# Step 1: Reserve 20% for final testing
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# Step 2: Take 25% of the remaining 80% for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,
    random_state=42,
    stratify=y_dev,
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

then we can use iteration to do Hyperparameter Selection:


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

candidate_k = [1, 3, 5, 7, 9]

for k in candidate_k:
    model = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=k),
    )
    model.fit(X_train, y_train)
    val_accuracy = model.score(X_val, y_val)

    print(f"k={k}, Validation Accuracy={val_accuracy:.3f}")

However, this kind of Validation Split may be heavily inluenced by randomness. This is why we have to introduce cross validation.


### 4.2.3 Cross-Validation (CV)

#### 4.2.3.1 K-Fold CV

Cross Validation is a method that make more use of Training Data while also reduce randomness. The most common CV is K-Fold Cross Validation. For example:


In [ ]:
from sklearn.model_selection import cross_val_score

model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5),
)

scores = cross_val_score(
    model,
    X_dev,
    y_dev,
    cv=5,
    scoring="accuracy",
)

print("Fold Accuracies:", scores)
print("Mean Accuracy:", scores.mean())
print("Std:", scores.std())

The whole process is mainly consisted of:

1. Divide the Development Data (`_dev`) into 5 folds;
2. Train with 4 folds each time, and the remaining 1 is left for validation;
3. Iterate 5 times to ensure that every Fold had been used as Validation Set;
4. Aggregate the Validation Scores.

One BIG IDEA here is that CV should include the whole Processing Pipeline. This is because `cross_val_score()` will do the fitting for each Fold, so `StandardScaler` only fit on the training proportion of the current division.


#### 4.2.3.2 Stratified K-Fold

For Classification, we wish to keep the original proportions if the sets. Scikit-learn automatically uses stratified fold, and does not disorder the samples. We can appoint it explicitly:


In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scores = cross_val_score(
    model,
    X_dev,
    y_dev,
    cv=cv,
    scoring="accuracy",
)

### 4.2.4 Classification Evaluation Metrics

#### 4.2.4.1 Confusion Matrix

Confusion Matrix is a 2*2 matrix which has the 4 elements bellow:

1. True Positive (TP): positive in prediction and reality;
2. True Nagative (TN): negative in prediction and reality;
3. False Positive (FP): positive in prediction, nagative in reality;
4. False Negative (FN): negative in prediction, positive in reality.

#### 4.2.4.2 Accuracy

With Confusion Matrix, Accuracy can be described as:

$$
Accuracy = \frac{TP+TN}{TP+TN+FP+FN}
$$

#### 4.2.4.3 Precision

Precision concerns on the positive predictions, and is often described as:

$$
Precision = \frac{TP}{TP+FP}
$$

#### 4.2.4.4 Recall

Recall indicates the proportion correctly predicted, in the positive samples in reality.

$$
Recall = \frac{TP}{TP+FN}
$$

#### 4.2.4.5 F1-Score

Precision focus on the model, while Recall focus on the reality. F1-Score is the Harmonic Mean of Precision and Recall:

$$
F1 = \frac{2\cdot Precision\cdot Recall}{Precision+Recall}
$$


### 4.2.5 Using Evaluation Metrics in Scikit-Learn

Let's use Validation Set as an example:


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5),
)

model.fit(X_train, y_train)
y_pred = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))

print("Classification Report:")
print(
    classification_report(
        y_val,
        y_pred,
        target_names=iris.target_names,
    )
)

For multiple-classification like `Iris`, `classification_report()` will use each class in turn as Positive Class to calculate the corresponding Precision, Recall, and F1-Score.


### 4.2.6 CV for Hyperparameter Tuning

Combining what we have learned:


In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score

candidate_k = [1, 3, 5, 7, 9]

results = {}

for k in candidate_k:
    model = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=k),
    )

    scores = cross_val_score(
        model,
        X_dev,
        y_dev,
        cv=5,
        scoring="accuracy",
    )

    results[k] = scores.mean()

best_k = max(results, key=results.get)

print("CV Results:", results)
print("Best k:", best_k)

and then use the whole development set to fit the model:


In [ ]:
final_model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=best_k),
)

final_model.fit(X_dev, y_dev)

print("Final Test Accuracy:", final_model.score(X_test, y_test))